# Steel Plate Analysis Technical Report

Spencer Mundel and Damon Lewis

CPSC 322 Fall 2025 - Final Project

## Introduction

We are trying to classify the type of defect present in a steel plate, given various measurements and properties of the plate.

Our overall findings were that KNN performed by far the best, achieving overall accuracy scores of 90%+, with other overall metrics being 80%+. All other classifiers performed significantly worse, and in particular had very poor recall, F1, and precision scores for minority classes.

## Dataset Description
|                | Steel Plate Dataset |
|----------------|---------------------|
| Instances      | 1941                |
| Unique Classes | 7                   |
| Attributes     | 27                  |

The dataset does not contain any missing values. Most of the data is continuous, with only the steel type values (`TypeOfSteel_A300` and `TypeOfSteel_A400`) being categorical true/false, which is represented as 0/1.

In [1]:
import copy
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from mysklearn.mypytable import MyPyTable
from mysklearn.myutils import *
from mysklearn.myclassifiers import *

/home/damon/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## Data Cleaning

Complete data cleaning steps can be found in the [Data Cleaning Notebook](data_cleaning.ipynb).

Only a small amount of cleaning was done. The classes were converted from one-hot encoding to categorical values to work better with our EDA and models. We also created a normalized version of the data using the z-score method for later use by PCA and KNN.

In [2]:
CLASSES = ["Pastry","Z_Scratch","K_Scratch","Stains","Dirtiness","Bumps","Other_Faults"]
CLASS_HEADER = "Class"

data = MyPyTable().load_from_file("input_data/plate-data-clean.csv")
features = copy.deepcopy(data.column_names)
features.remove(CLASS_HEADER)

## EDA

Complete EDA and all graphs can be seen in our [EDA Notebook](EDA.ipynb).

We first examined the distributions of classes to identify any class imbalance. We examined the distributions of all attributes to try and identify trends or potential obvious relationships.

One of our main forms of EDA was to create boxplots for each attribute, grouped by class. Our goal was to identify attributes that had distributions that varied significantly between classes, making them more useful and informative for classification.

### Summary Statistics

In [3]:
data.compute_summary_statistics(features).pretty_print()

attribute                    min               max              mid               avg           median
---------------------  ---------  ----------------  ---------------  ----------------  ---------------
X_Minimum                 0         1705              852.5             571.136          435
X_Maximum                 4         1713              858.5             617.964          467
Y_Minimum              6712            1.29877e+07      6.49719e+06       1.65068e+06      1.20413e+06
Y_Maximum              6724            1.29877e+07      6.49721e+06       1.65074e+06      1.20414e+06
Pixels_Areas              2       152655            76328.5            1893.88           174
X_Perimeter               2        10449             5225.5             111.855           26
Y_Perimeter               1        18152             9076.5              82.966           25
Sum_of_Luminosity       250            1.15914e+07      5.79583e+06  206312            19202
Minimum_of_Luminosity     0   

### Data Visualizations

![Class Occurrences](media/counts.png)

Figure 1: Occurrences for Each Class

![Attribute Distributions](media/dist.png)

Figure 2: Distributions for Each Attribute

![Boxplot X Minimum](media/boxplot_X_Minimum.png)

Figure 3: Boxplot for Attribute X Minimum, Grouped by Class

![Boxplot Log Areas](media/boxplot_LogOfAreas.png)

Figure 4: Boxplot for Attribute Log of Areas, Grouped by Class

![Boxplot Thickness](media/boxplot_Steel_Plate_Thickness.png)

Figure 5: Boxplot for Attribute Steel Plate Thickness, Grouped by Class

### Observations

As seen in Figure 1. We quickly noticed a significant class imbalance, with the "Other" class making up nearly half the dataset, and some classes having less than 100 instances. We were concerned that models would tend to favor the majority class, with low precision and recall for minority classes.

We also did not see many interesting trends in the attribute distributions in Figure 2. Most attributes are skewed to one side, or basically uniformly distributed. A few have some peaks or valleys, but nothing stood out as particularly useful for classification.

The boxplots in Figures 1 - 5 were some of the more informative visualizations we made. We identified 5 to 10 attributes that had noticeable differences in distributions between classes. All graphs have clearly separated medians, but they often have overlap in their interquartile ranges. Many also have many outliers that still complicate classification.

## Model Testing and Results

Our standard testing procedure involved 10-fold stratified cross-validation. Stratification was important to us because of the class imbalance in the dataset. We used overall and per-class accuracy, precision, recall, and F1 scores to evaluate model performance. For overall scores, we used macro-averaging to again better account for class imbalance.

While we were evaluating models, we were looking for high accuracy, with good precision, recall, and F1 scores across all classes. We were looking for models that could accurately classify our minority classes, especially the two smallest classes. Models with metrics above 80% across the board were our ideal.

In [26]:
normalized = MyPyTable().load_from_file("input_data/plate-data-normal.csv")
binned = MyPyTable(data.column_names, data.data)

for feature in features:
    if feature == "TypeOfSteel_A300" or feature == "TypeOfSteel_A400":
        continue

    col_values = data.get_column(feature)
    binned_col = equal_width_bin(col_values, 10)
    binned.replace_column(feature, binned_col)

### Decision Tree

The decision tree used basic equal-width binning for continuous attributes. There was no pruning, depth limit, or other hyperparameter tuning. All attributes and instances were used.

In [27]:
stratified_kfold_tester(MyDecisionTreeClassifier(len(features)),
                                binned,
                                CLASS_HEADER,
                                CLASSES,
                                10)

Setup complete
Confusion Matrix
                Pastry    Z_Scratch    K_Scratch    Stains    Dirtiness    Bumps    Other_Faults
------------  --------  -----------  -----------  --------  -----------  -------  --------------
Pastry               0            0           42         0            0        1             115
Z_Scratch            0            0          129         0            0        0              61
K_Scratch            0            0          338         0            0        8              45
Stains               0            0            0         0            0        8              64
Dirtiness            0            0            4         0            0       11              40
Bumps                0            0           48         0            0       26             328
Other_Faults         0            0          158         0            0       43             472
accuracy: 0.8373445204975345
precision: 0.16538487858712111
f1 score: 0.17689217635985477
recal

The overall accuracy was technically okay at 83.7%, but this is misleading. Overall precision, recall, and F1 are quite bad at around or below 20%. We can clearly see from the confusion matrix that the model is heavily favoring the majority class.

Per class metrics are very bad, with precision, recall, and F1 sometimes being 0 for several classes. The decision tree is clearly assuming that most instances belong to the majority class.

### KNN

Our full kNN testing and hyperparameter tuning process can be found in our [kNN Notebook](knn.ipynb).

We implemented kNN with Euclidean distance and majority voting. We tested k values from 1 to 6, and settled on k=3 as the smallest k with good performance. 

We experimented with normalized data (z-score) and PCA.

For our normalized data, we used manual feature selection based on our EDA boxplots. We initially tried to use Random Forest to find attributes with low entropy, but it was not working correctly, so we did not go forward with it. Instead, we experimented with PCA to reduce the dimensionality of our dataset. We settled on using the top 10 principal components, which captured about 90% of the variance in the data.

Our PCA implementation can be found in our [PCA Notebook](pca.ipynb).

Our final kNN model used k=3 and PCA with 10 components.

In [29]:
data_pca = MyPyTable().load_from_file("input_data/plate-data-pca.csv")

features = deepcopy(data_pca.column_names)
features.pop(features.index(CLASS_HEADER))

stratified_kfold_tester(MyKNeighborsClassifier(n_neighbors=3),
                                data_pca,
                                CLASS_HEADER,
                                CLASSES,
                                10)

Setup complete
Confusion Matrix
                Pastry    Z_Scratch    K_Scratch    Stains    Dirtiness    Bumps    Other_Faults
------------  --------  -----------  -----------  --------  -----------  -------  --------------
Pastry              62            6            1         0            4       37              48
Z_Scratch            2          141            8         0            2       11              26
K_Scratch            1            0          373         2            0        4              11
Stains               0            0            0        65            0        5               2
Dirtiness            2            2            0         0           44        5               2
Bumps               19           10            0         0            7      255             111
Other_Faults        21           36           20         8           11      150             427
accuracy: 0.9155074703760947
precision: 0.7101659485251988
f1 score: 0.7121041294808365
recall:

Our kNN model performed very strongly, achieving overall accuracy of 91.6%, with overall precision, recall, and F1 all above 70%. Looking further into per-class metrics, we can see about all metrics are above 50% for all classes, with many being above 70% and some even above 80%. This is quite good, even though it doesn't technically meet our ideal of 80%+ across the board.

Looking at the confusion matrix, while the model still has some tendency to favor the majority class, it does a much better job, even with our smallest classes.

### Random Forest
Equal width binned values were provided to the random forest here so decisions could be made on continuous features (which was the majority of the features in the original dataset). 5 of the best trees were selected out of 10 total trees generated, and each tree selected 5 random features.

In [30]:
stratified_kfold_tester(MyRandomForestClassifier(10, 5, 5),
                                binned,
                                CLASS_HEADER,
                                CLASSES,
                                10)

Setup complete
Confusion Matrix
                Pastry    Z_Scratch    K_Scratch    Stains    Dirtiness    Bumps    Other_Faults
------------  --------  -----------  -----------  --------  -----------  -------  --------------
Pastry               0            0           42         0            0       10             106
Z_Scratch            0            0          129         0            0        1              60
K_Scratch            0            0          338         0            0       10              43
Stains               0            0            0         0            0        8              64
Dirtiness            0            0            4         0            0       13              38
Bumps                0            0           48         0            0       29             325
Other_Faults         0            0          158         0            0       48             467
accuracy: 0.8370501214396114
precision: 0.162455084132241
f1 score: 0.17803346944421802
recall:

The overall model accuracy scored at around 83%, but precision, accuracy, and recall were all below 30%, likely indicating an over selection of the more frequent classes (as the model could not even identify any class with a count under 200). 

#### Clustered Random Forest
Clustering is applied here on each of the random attributes that are selected for each decision tree in the random forest. Otherwise, the behavior is the same as a random forest.

Hyperparameter tuning was done in the form of trial and error in terms of the number of clusters for each feature. 2 clusters per feature seemed to yield the best results; this is what is done here. Otherwise, the same N, M, and F values are used for the clustered random forest as in the other random forest used on the dataset; this way the clustered and non-clustered random forests can be compared equally to see what the impact of clustering the data is.

In [31]:
stratified_kfold_tester(MyClusteredRandomForestClassifier(10, 5, 5, 2),
                                data,
                                CLASS_HEADER,
                                CLASSES,
                                10)

Setup complete
Confusion Matrix
                Pastry    Z_Scratch    K_Scratch    Stains    Dirtiness    Bumps    Other_Faults
------------  --------  -----------  -----------  --------  -----------  -------  --------------
Pastry               0           11           12         0            0       27             108
Z_Scratch            0          138           13         0            0        1              38
K_Scratch            0            0          257         0            0       27             107
Stains               0            0            0         0            0        3              69
Dirtiness            0            0            2         0            0        6              47
Bumps                0           16            9         2            0       95             280
Other_Faults         3           36           25         0            0       71             538
accuracy: 0.8656068300581438
precision: 0.3372900305720051
f1 score: 0.32999637529408676
recall

The usage of clustering improved the ability of the model to pick up on less frequent classes, but it still completely missed "Pastry", "Z_Scratch", and "K_Scratch". Hence, there was a significant increase of about 10-20 percentage points in recall, precision, and F1 score, but only a marginal improvement from 84% to 87% was observed in overall accuracy.

### Results and Conclusion
Overall, KNN significantly outperforms decision tree and random forest with this dataset in general, although further work with normalization and PCA narrowed the gap in terms of overall accuracy. The dataset had an uneven distribution of class counts making it more difficult to find specific rules when fitting random forest and a decision tree.

We implemented some of the ideas that we had after our presentation in order to improve classifier performance. Z-Score normalization and PCA were used to improve the KNN classifier; overall accuracy increased from 71% to 92%. Additionally, clustering was used with random forest which significantly improved precision, recall, and F1 score overall while only moderately improving overall accuracy. At first, only equal width binning and min-max normalization were used, but such techniques are vulnerable to large outliers in feature sets. Clustering and z-score normalization helped reduce this issue by separating the "normal" feature values from the "outlier" feature values. 

### Contributions
Spencer: utilities, EDA, stratified k-fold, data mining, presentation, clustering

Damon: random forest, data mining, metrics, presentation, PCA and KNN hyperparameter tuning

### Citations
M. Buscema, S. Terzi, and W. Tastle. "Steel Plates Faults," UCI Machine Learning Repository, 2010. [Online]. Available: https://doi.org/10.24432/C5J88N.